# OASIS-2 Multimodal Training Pipeline

This notebook is the entry point for the **Multimodal Alzheimer's AI Monitoring Platform**.
It fuses 3D brain MRI scans with clinical metadata (Age, Sex, MMSE) using the `OASISMultimodalDenseNet` architecture.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

# Constants for your environment
DRIVE_ROOT = Path('/content/drive/MyDrive/Cerebrasensecloud')
OASIS2_BUNDLE_ROOT = DRIVE_ROOT / 'OASIS-2'
RUNTIME_ROOT = DRIVE_ROOT / 'backend_runtime'
RUN_NAME = 'oasis2_multimodal_v1' # Changed for fresh multimodal run

for name, path in {
    'DRIVE_ROOT': DRIVE_ROOT,
    'OASIS2_BUNDLE_ROOT': OASIS2_BUNDLE_ROOT,
    'RUNTIME_ROOT': RUNTIME_ROOT,
}.items():
    print(f"{name}: {'[OK]' if path.exists() else '[MISSING]'} {path}")

In [ ]:
import shutil
import subprocess
from pathlib import Path

REPO_ROOT = Path('/content/cerebrasense')
BACKEND_ROOT = REPO_ROOT / 'alz_backend'
REPO_URL = 'https://github.com/Billrichard209/cerebrasense.git'
REQUIRED_COMMIT = '8da5ed6' # Latest multimodal integration commit

def run_checked(cmd, *, cwd=None, label=None):
    print(f"RUNNING {label or cmd[0]}: {' '.join(cmd)}", flush=True)
    completed = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout, flush=True)
    if completed.stderr:
        print(completed.stderr, flush=True)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed ({label or cmd[0]}): {' '.join(cmd)}")
    return completed

# Clean up any stale repo clones
for stale_root in [Path('/content/cerebrasense'), Path('/content/Cerebrasense-')]:
    if stale_root.exists():
        shutil.rmtree(stale_root)

run_checked(['git', 'clone', REPO_URL, str(REPO_ROOT)], cwd='/content', label='git-clone')
run_checked(['git', 'checkout', 'main'], cwd=REPO_ROOT, label='git-checkout-main')
run_checked(['python3', '-m', 'pip', 'install', '-r', str(BACKEND_ROOT / 'requirements-colab.txt')], cwd=REPO_ROOT, label='pip-install')

repo_commit = run_checked(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, label='git-rev-parse').stdout.strip()
print(f'Active commit: {repo_commit}')
if repo_commit[:7] != REQUIRED_COMMIT[:7]:
    print(f"WARNING: Current commit {repo_commit[:7]} does not match expected {REQUIRED_COMMIT[:7]}")

### Start Multimodal Training
This command will initiate a 50-epoch training run using the `densenet121_multimodal` architecture. It automatically fuses MRI volumes with clinical metadata (Age, Sex, MMSE) extracted from your OASIS-2 bundle.

In [ ]:
import os
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected! Change runtime type to T4 GPU before proceeding.")

os.chdir(BACKEND_ROOT)

!python scripts/train_oasis2_colab.py \
    --project-root {BACKEND_ROOT} \
    --runtime-root {RUNTIME_ROOT} \
    --bundle-root {OASIS2_BUNDLE_ROOT} \
    --run-name {RUN_NAME} \
    --epochs 50 \
    --batch-size 2 \
    --gradient-accumulation-steps 4 \
    --num-workers 2 \
    --image-size 96 96 96 \
    --learning-rate 2e-4 \
    --weight-decay 0.01 \
    --scheduler cosine \
    --weighted-sampling \
    --seed 42 \
    --split-seed 42 \
    --device auto \
    --config configs/oasis2_train.yaml

### Result Summary
Run the cell below after training completes to see your final metrics and paths to the saved model checkpoints.

In [ ]:
import json
import pandas as pd
from pathlib import Path

run_root = RUNTIME_ROOT / 'outputs' / 'runs' / 'oasis2' / RUN_NAME
summary_path = run_root / 'reports' / 'colab_run_summary.json'
metrics_path = run_root / 'metrics' / 'epoch_metrics.csv'

if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(f"\nTraining Summary for {RUN_NAME}:")
    print(f"- Best Epoch: {summary.get('best_epoch')}")
    print(f"- Best Val AUROC: {summary.get('best_monitor_value'):.4f}")
    print(f"- Best Checkpoint: {summary.get('best_checkpoint')}")

if metrics_path.exists():
    df = pd.read_csv(metrics_path)
    print("\nLast 5 Epochs:")
    print(df.tail())
else:
    print("\nNo metrics found. Training may still be in progress or failed.")